In [2]:
from deconvolawrence.meta_processing import *
from deconvolawrence import ms_plotter_tools as msp
import sys
import os
# matplotlib.rcParams['font.family'] = 'Arial'


ModuleNotFoundError: No module named 'deconvolawrence'

In [ ]:
folder = r"D:\240724 simplified doe attempt 1\files"

os.listdir(folder)

# Automated deconvolution and quantification of mass spectra 
*Author - Lawrence Collins*

It is recommended to restart your computer before devconvolving large dataets (40+) as it is quite RAM intensive.
## Step 1: Load input file

Make sure to use double backslash. 
e.g. ``` path = "D:\\my mass spec folder\\my mass spec experiment\\mass_spec_input_file.xlsx" ```

### Setting up the input file

Sheet 1 of input file contains the directory folder of your mass spectra and the unidec configuration parameters. 

You can also add masses to detect and a corresponding colour. These are done by stipulating **Species** + *name* and **Color** + *name* in the Parameter column, followed by the desired mass or colour in the Input column.

A list of accepted colour names can be found [here](https://matplotlib.org/stable/gallery/color/named_colors.html)


**Example input directory**

| Parameter | Input | Comments |
| --- | --- |--- |
|Directory|D:\mass spec\protein labelling|
|Species Protein| 48127|
|Color Protein| orange|
|Start Scan|	490	|
|End Scan|540	|
|Tolerance (Da)	|10	|Peak matching tolerance |
|Config masslb	|15000|	Deconvolution window low mass|
|Config massub	|50000|	Deconvolution window high mass|
|Config massbins|	1|Mass bins for deconvolution - sample mass every|
|Config peakwindow|	10|	|
|Config peakthresh|0.05|	|
|Config minmz|	700|	m/z lower bounds (defaults to 0)|
|Config maxmz|		|m/z upper bounds (defaults to 10e12)|
|Config startz|	1	| |
|Config endz|	100	| |
|Config numz|	100	| |
|Config numit|	60	|number of iterations of deconvolution algorithm|

### Defining variables of different spectra

Each file within the directory can be linked to custom variables defined in a second sheet of the input directory. This comes in handy if wanting to filter data or perform analyses/comparison on subsets of your experiment. 

Any column names can be defined aside from 'Name' in column 0. Name corresponds to your filename (can be partial match).

Make sure var_ids=True ```load_input_file(var_ids = True)```  

**Example variables table**

|Name|	Peptide	|Catalyst|Time|
| --- | --- | --- | --- |
|240126 DoE Nexp 1	|1	|1	|1|
|240126 DoE Nexp 2	|2	|13	|3|
|240126 DoE Nexp 3	|1	|25	|3|
|240126 DoE Nexp 4	|3	|25	|1|
|240126 DoE Nexp 5	|2	|13	|5|
|240126 DoE Nexp 6	|3	|1	|3|
|240126 DoE Nexp 7	|3	|13	|1|
|240126 DoE Nexp 8	|2	|1	|5|
|240126 DoE Nexp 9	|1	|25	|5|




In [ ]:
# path = "Meta2_input_file_v3.xlsx" # path to input file

# path = "Alexandra_MS_input_file.xlsx" # path to input file

# path = r"240308 DoE input file.xlsx"
path = r"C:\Users\cm19ljc\OneDrive VERSION 2\OneDrive - University of Leeds\RESEARCH\PROJECT\Lab\Labelling\240724 DoE input file.xlsx"
path = r"C:\Users\cm19ljc\OneDrive VERSION 2\OneDrive - University of Leeds\RESEARCH\PROJECT\Lab\Labelling\240724 Simplified DoE input file.xlsx"

eng = Meta2() # load engine

eng.load_input_file(path, unzip=False, clearhdf5=True, var_ids=True) # load input file and run deconvolution

eng.on_unidec() # run deconvolution

In [ ]:
eng.results1

In [ ]:
eng.results_df

## Step 2: Plotting mass spectra

Mass spectra can be plotted separately and exported to *UniDec_Figures_and_Files* within your directory. 

### Useful commands

You can plot the deconvolved spectrum ```attr = "massdat"``` or the raw spectrum ```attr = "data2"``` (make sure to change the xlabel to m/z if plotting the raw spectrum ```xlabel = "m/z"```)

```window= [12000, 14000] ``` A mass window over which to plot over 

```legend = True``` to add a legend of the masses found within the spectra

```show_peaks = True``` to pick the peaks that correspond to defined species from the input file found in the spectrum

```fmt="svg"``` to plot in vector format (editable in photoshop) or ```fmt = 'png'``` for image files

### Combining spectra to a single figure 
Spectra can be stacked for nice made-for-publication figures. This can be combined with any defined variables within the second sheet of your input file to separate desired mass spectra to be stacked - e.g. if plotting a time course. 

To separate groups of spectra by variables from the second sheet of the input file into combined figures, define ```groupby = [variable_x]``` and ```combine = True``` (can be grouped by more than one variable by writing a list of variables into groupby).

In [ ]:
cm = 1/2.54  # centimeters in inches

width = 15.92

spectra = eng.eng.data.spectra

figsize = (width*cm, width*cm*(1/3)*len(spectra))



# add in zoomed function that plots over window using min and max masses contained in masslist 
msp.plot_spectra_separate(spectra, attr = "massdat", xlabel = "Mass [Da]", 
                          export=True, c='black',lw=0.7,window=[None, None],
                         show_peaks=False,legend=False, directory =eng.directory, fmt='png',
                         figsize=figsize)

# msp.plot_spectra_separate(spectra, attr = "data2", xlabel = "m/z", export=False)


# eng.group_spectra("GVSEYG")

In [ ]:
# eng.plot_spectra(combine=True, groupby='GVSEYG',show_titles=False)
spectra[0].name[:-2]

In [ ]:
sorted_spectra = []
df = eng.results_df
spectra = eng.eng.data.spectra


for row in df.iterrows():
    name = row[1].Name
    print(name)
    for s in spectra:
        if s.name[:-2] ==name:
            named = s
    sorted_spectra.append(named)
    # named = [s for s in spectra if s.name[:-2] == name][0]
    # sorted_spectra.append(named)


In [ ]:
df

In [ ]:
msp.plot_spectra_combined2(sorted_spectra,window = [40000, 45000], export = False, figsize=(3,9), show_titles=False)
plt.show()

In [ ]:
df = eng.results1.copy()

df = df[df.Temp == 25]
df = df[df.Time==30]
df

## Step 3: Quantification and analysis
A variety of useful statistical plotting tools taken from [Seaborn](https://seaborn.pydata.org/) and [Matplotlib](https://matplotlib.org/) can used to display your quantified data. Defined variables can be used to filter and select subsets of the data. 

In [ ]:
x="Catalyst"
y='Time'
z='Substrate'
c = 'Percentage_Labelling'
on=['CTB-LPETGVSEYG',"CTB-LPET" ]
df=eng.results1
df=eng.results1[df.Temp==4]
msp.plot3d(x, y, z, df, on, on_column='Label',c=c)

df=eng.results1[eng.results1.Temp==25]
msp.plot3d(x, y, z, df, on, on_column='Label',c=c)

In [ ]:
x="Time"
y='Percentage_Labelling'
z='GVSEYG'
on=['CTB-LPETGVSEYG', "CTB-LPET"]
on_column='Label'
# on=None
# on_column=None
msp.plot_data(df=eng.results1, x=x, y=y, on=on, hue=on_column, on_column=on_column,palette=eng.colors_dict)


In [ ]:
import matplotlib 
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'serif'

p1=sns.relplot(eng.results1, x='Time', y='Percentage_Labelling', hue='Label',col='Catalyst', row='Substrate', palette=eng.colors_dict,legend="full")
ax = p1.axes[0,0]

ax.set_ylim(0, 100)

plt.show()